# Step 2-1. 전문가 Agent 하나 만들기: 의약화학 전문가
오늘의 목표는 AWS 블로그의 SK바이오팜 사례처럼 **신약 후보물질을 여러 전문가 Agent가 함께 검토하는 시스템**을 만드는 것입니다. 먼저 전문가 한 명부터 만듭니다.

## 오늘의 전체 그림
1. **Step 2-1** 공개 데이터베이스를 조회하는 도구와, 그 도구를 쓰는 전문가 Agent 한 명
2. **Step 2-2** 전문가 세 명과 지휘자(Orchestrator)로 Multi-Agent 만들기
3. **Step 2-3** 내 주제로 팀을 만들어 웹 화면으로 띄우기 (완성 예시: 흉부 엑스레이 판독 보조 팀)
4. **Step 2-4** 큰 모델(스승)의 판단을 작은 모델(제자)에게 가르치기 (Distillation, 증류)
5. **Step 2-5, 2-6** (심화) 증류를 한 단계씩 뜯어 보기: 학습 자료 만들기, 학습시키기

## 이 노트북의 핵심 개념
- **Tool(도구)**: Agent가 부를 수 있는 함수. 여기서는 PubChem, PubMed 같은 공개 데이터베이스 조회
- **Strands Agents**: AWS가 공개한 Agent 제작 도구. 블로그의 SK바이오팜 팀이 쓴 것과 같습니다
- 모델은 내 맥의 Ollama에서 돕니다. API 키도 요금도 없습니다

In [ ]:
# 필요한 패키지를 설치합니다. 처음에는 1~2분 걸립니다.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "strands-agents[ollama]", "rdkit", "requests", "ollama"], check=True)
print("✅ 패키지 준비 완료. 다음 셀로 넘어가세요.")

## 1. 설정
전문가 역할에는 도구를 정확히 쓰는 모델이 필요해서 `gemma4:e4b`를 씁니다. 없으면 여기서 내려받습니다.

In [ ]:
import os, sys
import ollama

sys.path.insert(0, os.path.abspath(".."))   # 저장소 맨 위의 drug_tools.py를 불러오기 위해

EXPERT_MODEL = "gemma4:e4b"    # 전문가와 지휘자 역할을 하는 모델 (약 9.5GB)
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")   # GPU 서버를 쓰면 그 주소로 바뀝니다

installed = [m.model for m in ollama.list().models]
if not any(m == EXPERT_MODEL or m.startswith(EXPERT_MODEL + ":") for m in installed):
    print(f"⬇️  {EXPERT_MODEL} 내려받는 중... (약 9.5GB, 처음 한 번만)")
    ollama.pull(EXPERT_MODEL)
print(f"✅ 모델 준비 완료: {EXPERT_MODEL}  (Ollama 주소: {OLLAMA_HOST})")

## 2. 도구를 직접 써 보기
`drug_tools.py`에 공개 데이터베이스를 조회하는 함수 여섯 개가 있습니다. Agent에게 주기 전에 우리가 먼저 불러 봅니다.

| 함수 | 하는 일 | 데이터 |
|---|---|---|
| `compound_info` | 화합물 기본 정보와 SMILES(구조식) | PubChem |
| `molecule_properties` | 분자량, 지용성, Lipinski 규칙 계산 | 내 컴퓨터에서 계산(RDKit) |
| `similar_compounds` | 구조가 비슷한 화합물 | PubChem |
| `pubmed_search` | 논문 검색 | PubMed |
| `clinical_trials` | 임상시험 검색 | ClinicalTrials.gov |
| `adverse_events` | 보고된 부작용 | openFDA |

> 블로그에서는 이 단계에 TxGemma라는 신약 전용 모델을 썼습니다. GPU 서버가 필요해서 오늘은 계산 라이브러리(RDKit)로 대신합니다.

In [ ]:
import drug_tools as dt

# TODO: 다른 약 이름(영문)으로 바꿔 보세요! 예: metformin, warfarin, atorvastatin
DRUG = "ibuprofen"

info = dt.compound_info(DRUG)
print("📦 PubChem:", info["formula"], "| 분자량", info["molecular_weight"], "| SMILES", info["smiles"])

props = dt.molecule_properties(info["smiles"])
print("🧪 계산한 특성:")
for k in ["molecular_weight", "logp", "h_bond_donors", "h_bond_acceptors", "tpsa", "qed", "lipinski_pass"]:
    print(f"   {k}: {props[k]}")

In [ ]:
print("📚 PubMed 논문:")
for p in dt.pubmed_search(f"{DRUG} safety", 3)["items"]:
    print(f"   {p['year']}  {p['title'][:80]}")

print("\n🏥 임상시험:")
for t in dt.clinical_trials(DRUG, 3)["items"]:
    print(f"   {t['nct_id']}  {t['phase']:<10} {t['title'][:60]}")

print("\n⚠️ 보고된 부작용 (건수):")
for a in dt.adverse_events(DRUG, 5)["items"]:
    print(f"   {a['reports']:>7}  {a['reaction']}")

## 3. 도구를 Agent에게 주기
함수 위에 `@tool`을 붙이면 Agent가 부를 수 있는 도구가 됩니다. 함수 설명글(docstring)을 모델이 읽고 언제 쓸지 판단합니다. 1일차의 MCP와 같은 원리입니다.

In [ ]:
from strands import Agent, tool
from strands.models.ollama import OllamaModel


@tool
def compound_info(name: str) -> dict:
    """화합물의 영문 이름으로 PubChem에서 기본 정보와 SMILES 구조식을 찾는다."""
    return dt.compound_info(name)


@tool
def molecule_properties(smiles: str) -> dict:
    """SMILES 구조식으로 분자량, 지용성(logP), 수소결합 수, Lipinski 규칙 통과 여부를 계산한다."""
    return dt.molecule_properties(smiles)


@tool
def similar_compounds(smiles: str) -> dict:
    """SMILES 구조식과 구조가 비슷한 화합물을 PubChem에서 찾는다."""
    return dt.similar_compounds(smiles, 5)


print("✅ 도구 3개 준비 완료")

## 4. 전문가 Agent 만들기
Agent는 **모델 + 역할 설명(System Prompt) + 도구**입니다. 블로그의 "Walter White(화학 전문가)"에 해당합니다.

In [ ]:
def make_model():
    return OllamaModel(host=OLLAMA_HOST, model_id=EXPERT_MODEL, temperature=0.2)


CHEMIST_PROMPT = """너는 15년 경력의 의약화학 전문가다.
- 도구로 화합물 정보를 찾고, 물리화학적 특성을 계산하고, 구조가 비슷한 화합물을 확인한다.
- 도구 결과에 있는 숫자만 쓴다. 없는 내용은 지어내지 않는다.
- 한국어로, 소제목 3개 이내, 전체 10줄 이내로 답한다.
- 마지막 줄에 확신 정도(높음, 보통, 낮음)를 적는다."""

chemist = Agent(model=make_model(), system_prompt=CHEMIST_PROMPT,
                tools=[compound_info, molecule_properties, similar_compounds], callback_handler=None)
print("✅ 의약화학 전문가 Agent 준비 완료")

## 5. 물어보기
처음 부를 때는 모델을 메모리에 올리느라 30초쯤 걸릴 수 있습니다.

In [ ]:
def used_tools(agent):
    """Agent가 이번 대화에서 부른 도구 이름을 순서대로 돌려준다."""
    return [b["toolUse"]["name"] for m in agent.messages for b in m.get("content", []) if "toolUse" in b]


# TODO: 질문을 바꿔 보세요!
question = f"{DRUG}의 구조와, 먹는 약으로서의 성질을 평가해 줘"

answer = chemist(question)
print("👤 질문:", question)
print("🔧 부른 도구:", " → ".join(used_tools(chemist)))
print("🧑‍🔬 답:\n", str(answer).strip())

## 정리
- 전문가 Agent 하나는 "역할 설명 + 도구 몇 개"로 만듭니다.
- 도구가 진짜 데이터를 가져오기 때문에, 모델이 숫자를 지어낼 여지가 줄어듭니다.
- 다음은 전문가를 세 명으로 늘리고, 누구에게 물을지 정하는 지휘자를 둡니다. → Step 2-2